In [32]:
"""
build_marts.py — Студент 4
Сборка финальных витрин данных + ML-модель (Random Forest)
Запуск: python scripts/build_marts.py
"""

import os
import ast
import warnings
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# 0. ПАПКИ
# ─────────────────────────────────────────────
os.makedirs('output_mart', exist_ok=True)
os.makedirs('reports',     exist_ok=True)

# ─────────────────────────────────────────────
# 1. ЗАГРУЗКА ДАННЫХ
# ─────────────────────────────────────────────
print("=== Загрузка данных ===")

fact_dtp  = pd.read_csv('output/fact_dtp.csv')
fact_part = pd.read_csv('output/fact_participant.csv')
fact_veh  = pd.read_csv('output/fact_vehicle.csv')
weather   = pd.read_csv('output_w/feat_weather_dtp_first4600lines.csv')
spatial   = pd.read_csv('output_spatial/feat_spatial_dtp.csv')

print(f"fact_dtp:  {len(fact_dtp):,} строк")
print(f"fact_part: {len(fact_part):,} строк")
print(f"fact_veh:  {len(fact_veh):,} строк")
print(f"weather:   {len(weather):,} строк  ← покрытие {len(weather)/len(fact_dtp)*100:.1f}%")
print(f"spatial:   {len(spatial):,} строк  ← покрытие {len(spatial)/len(fact_dtp)*100:.1f}%")

# ─────────────────────────────────────────────
# 2. ДЖОЙН В mart_dtp_enriched
# ─────────────────────────────────────────────
print("\n=== Джойн ===")

df = (
    fact_dtp
    .merge(weather, on='dtp_id', how='left')
    .merge(spatial,  on='dtp_id', how='left')
)

assert len(df) == len(fact_dtp), \
    f"ОШИБКА: потеряли строки при джойне! {len(df)} != {len(fact_dtp)}"

print(f"Строк после джойна: {len(df):,} ✓")
print(f"Колонок: {df.shape[1]}")

# ─────────────────────────────────────────────
# 3. ВРЕМЕННЫЕ ПРИЗНАКИ
# ─────────────────────────────────────────────
print("\n=== Временные признаки ===")

df['moment_date']     = pd.to_datetime(df['moment_date'])
df['hour']            = pd.to_datetime(df['moment_time'], errors='coerce').dt.hour
df['month']           = df['moment_date'].dt.month
df['day_of_week_dtp'] = df['moment_date'].dt.dayofweek  # 0=пн, 6=вс

# Время суток
def get_time_of_day(hour):
    if pd.isna(hour):
        return None
    h = int(hour)
    if   6  <= h < 10: return 'Утро'
    elif 10 <= h < 17: return 'День'
    elif 17 <= h < 22: return 'Вечер'
    else:              return 'Ночь'

df['time_of_day']  = df['hour'].apply(get_time_of_day)

df['is_night_dtp'] = df['hour'].apply(
    lambda h: 1 if pd.notna(h) and (int(h) >= 22 or int(h) < 6) else 0
)
df['is_rush_hour'] = df['hour'].apply(
    lambda h: 1 if pd.notna(h) and (7 <= int(h) < 10 or 17 <= int(h) < 20) else 0
)

# Сезон из даты (резерв, если weather не покрывает все строки)
season_map = {
    12:'Зима', 1:'Зима',  2:'Зима',
    3:'Весна', 4:'Весна', 5:'Весна',
    6:'Лето',  7:'Лето',  8:'Лето',
    9:'Осень', 10:'Осень',11:'Осень'
}
# Если season из weather есть — оставляем, иначе считаем сами
if 'season' not in df.columns:
    df['season'] = df['month'].map(season_map)
else:
    df['season'] = df['season'].fillna(df['month'].map(season_map))

print("Временные признаки: ✓")

# ─────────────────────────────────────────────
# 4. ЦЕЛЕВАЯ ПЕРЕМЕННАЯ
# ─────────────────────────────────────────────
df['severity'] = (df['dead_count'] > 0).astype(int)

print(f"\nРаспределение severity:")
print(df['severity'].value_counts())
print(f"Доля с гибелью: {df['severity'].mean():.3f}")

# ─────────────────────────────────────────────
# 5. КЛАССИФИКАТОР ТИПОВ ДТП
# ─────────────────────────────────────────────
print("\n=== Классификатор типов ДТП ===")
print(df['type_name'].value_counts().to_string())

group_map = {
    'Столкновение':             'Столкновение',
    'Наезд на пешехода':        'Наезд на пешехода',
    'Наезд на препятствие':     'Наезд на препятствие',
    'Наезд на стоящее т/с':    'Наезд на стоящее ТС',
    'Опрокидывание':            'Опрокидывание',
    'Наезд на велосипедиста':   'Наезд на велосипедиста',
    'Падение пассажира':        'Падение пассажира',
    'Съезд с дороги':           'Съезд с дороги',
    'Наезд на лицо, не являющееся участником дорожного движения': 'Прочее',
}

df['type_group'] = df['type_name'].map(group_map).fillna('Прочее')
print("\nГруппы ДТП:")
print(df['type_group'].value_counts().to_string())

# ─────────────────────────────────────────────
# 6. ФЛАГИ НАРУШЕНИЙ ПДД (из fact_part)
# ─────────────────────────────────────────────
print("\n=== Флаги нарушений ПДД ===")

def parse_violations(val):
    """Строку-список превращает в настоящий список Python"""
    if pd.isna(val) or str(val).strip() in ('[]', 'nan', ''):
        return []
    try:
        return ast.literal_eval(str(val))
    except Exception:
        return []

fact_part['violations_list'] = fact_part['main_pdd_derangements'].apply(parse_violations)

# Смотрим все уникальные нарушения (выводим первые 30)
all_viols = fact_part['violations_list'].explode().dropna().unique()
print("Примеры нарушений:")
print(sorted(all_viols)[:30])

def has_keyword(viol_list, keywords):
    return int(any(
        any(kw.lower() in v.lower() for kw in keywords)
        for v in viol_list
    ))

flags = (
    fact_part
    .groupby('dtp_id')['violations_list']
    .apply(list)
    .reset_index()
)
flags.columns = ['dtp_id', 'all_violations']
flags['all_flat'] = flags['all_violations'].apply(
    lambda lists: [v for sublist in lists for v in sublist]
)

flags['has_speed_violation']    = flags['all_flat'].apply(
    lambda x: has_keyword(x, ['скорост']))
flags['has_priority_violation'] = flags['all_flat'].apply(
    lambda x: has_keyword(x, ['приоритет', 'сигнал', 'светофор', 'регулирован']))
flags['has_alcohol_violation']  = flags['all_flat'].apply(
    lambda x: has_keyword(x, ['опьянен', 'алкогол', 'наркот']))
flags['has_pedestrian_viol']    = flags['all_flat'].apply(
    lambda x: has_keyword(x, ['пешеход']))
flags['has_distance_viol']      = flags['all_flat'].apply(
    lambda x: has_keyword(x, ['дистанц']))
flags['has_maneuver_viol']      = flags['all_flat'].apply(
    lambda x: has_keyword(x, ['маневр']))

# Молодой водитель (< 25 лет) и новичок (стаж < 2 лет)
driver_df = fact_part[fact_part['part_type_code'] == 'Водитель'].copy()
driver_df['person_age']            = pd.to_numeric(driver_df['person_age'],            errors='coerce')
driver_df['driver_service_length'] = pd.to_numeric(driver_df['driver_service_length'], errors='coerce')

age_flags = driver_df.groupby('dtp_id').agg(
    has_young_driver  = ('person_age',            lambda x: int((x < 25).any())),
    has_novice_driver = ('driver_service_length',  lambda x: int((x < 2).any())),
).reset_index()

VIOL_COLS = [
    'dtp_id', 'has_speed_violation', 'has_priority_violation',
    'has_alcohol_violation', 'has_pedestrian_viol',
    'has_distance_viol', 'has_maneuver_viol'
]
df = df.merge(flags[VIOL_COLS], on='dtp_id', how='left')
df = df.merge(age_flags,        on='dtp_id', how='left')

print("Флаги нарушений: ✓")
print(df[['has_speed_violation','has_priority_violation',
          'has_alcohol_violation']].sum())

# ─────────────────────────────────────────────
# 7. ФЛАГИ ТИПОВ ТС (из fact_veh)
# ─────────────────────────────────────────────
print("\n=== Флаги типов ТС ===")
print(fact_veh['transport_type_name'].value_counts().to_string())

# Уточни значения под свои данные если нужно
truck_types = ['Грузовой']
bus_types   = ['Автобус']
taxi_types  = ['Такси']
moto_types  = ['Мотоцикл', 'Мопед']
share_types = ['Каршеринг']

veh_flags = (
    fact_veh
    .groupby('dtp_id')['transport_type_name']
    .apply(list)
    .reset_index()
)
veh_flags.columns = ['dtp_id', 'veh_types']

veh_flags['has_truck']      = veh_flags['veh_types'].apply(lambda x: int(any(t in truck_types for t in x)))
veh_flags['has_bus']        = veh_flags['veh_types'].apply(lambda x: int(any(t in bus_types   for t in x)))
veh_flags['has_taxi']       = veh_flags['veh_types'].apply(lambda x: int(any(t in taxi_types  for t in x)))
veh_flags['has_motorcycle'] = veh_flags['veh_types'].apply(lambda x: int(any(t in moto_types  for t in x)))
veh_flags['has_carsharing'] = veh_flags['veh_types'].apply(lambda x: int(any(t in share_types for t in x)))

VEH_COLS = ['dtp_id','has_truck','has_bus','has_taxi','has_motorcycle','has_carsharing']
df = df.merge(veh_flags[VEH_COLS], on='dtp_id', how='left')

print("Флаги ТС: ✓")

# ─────────────────────────────────────────────
# 8. ПРОВЕРКА SPATIAL-ПРИЗНАКОВ
# ─────────────────────────────────────────────
print("\n=== Покрытие spatial-признаков ===")

spatial_cols = [
    'has_traffic_light', 'has_crosswalk', 'has_bus_stop',
    'has_camera', 'is_intersection', 'is_open_road',
    'distance_to_intersection_m', 'distance_to_crosswalk_m',
    'distance_to_bus_stop_m', 'poi_count_100m',
    'traffic_lane_amount', 'lane_width_m', 'is_multilane'
]
# Только те колонки, которые реально есть в df
spatial_cols = [c for c in spatial_cols if c in df.columns]
missing_pct  = df[spatial_cols].isnull().mean().sort_values(ascending=False)
print(missing_pct.to_string())

# ─────────────────────────────────────────────
# 9. ML: RANDOM FOREST + DECISION TREE
# ─────────────────────────────────────────────
print("\n=== Обучение ML-модели ===")

# Только признаки, которые есть в df
CANDIDATE_FEATURES = [
    # Время
    'hour', 'day_of_week_dtp', 'is_night_dtp', 'is_rush_hour',
    # Тип ДТП
    'type_code',
    # Нарушения
    'has_speed_violation', 'has_priority_violation',
    'has_alcohol_violation', 'has_pedestrian_viol',
    'has_distance_viol', 'has_maneuver_viol',
    # Участники
    'has_young_driver', 'has_novice_driver',
    # ТС
    'has_truck', 'has_bus', 'has_taxi', 'has_motorcycle', 'has_carsharing',
    # Погода
    'temp_c', 'ice_risk', 'bad_weather_flag', 'is_dark', 'wind_speed_ms',
    # Инфраструктура
    'has_traffic_light', 'has_crosswalk', 'has_camera',
    'is_intersection', 'is_multilane', 'traffic_lane_amount', 'distance_to_bus_stop_m', 'poi_count_100m',
]

# Берём только те признаки, которые есть в df
FEATURES = [f for f in CANDIDATE_FEATURES if f in df.columns]
TARGET   = 'severity'

print(f"Признаков для модели: {len(FEATURES)} из {len(CANDIDATE_FEATURES)}")

model_df = df[FEATURES + [TARGET]].copy()
model_df = model_df.dropna(subset=[TARGET])

# Принудительно конвертируем ВСЕ колонки в числа
for col in FEATURES:
    model_df[col] = pd.to_numeric(model_df[col], errors='coerce')

# Проверяем где остались NaN после конвертации
nan_counts = model_df[FEATURES].isnull().sum()
print("NaN по признакам после конвертации:")
print(nan_counts[nan_counts > 0])

# Заполняем медианой
for col in FEATURES:
    median_val = model_df[col].median()
    model_df[col] = model_df[col].fillna(median_val)

# Финальная проверка — не должно быть ни одного NaN
assert model_df[FEATURES].isnull().sum().sum() == 0, \
    "Остались NaN! Смотри вывод выше."

print("NaN после заполнения: 0 ✓")

X = model_df[FEATURES].astype(float)  # явно float
y = model_df[TARGET].astype(int)

print(f"Размер датасета: {len(X):,} строк")
print(f"Доля гибелей (класс 1): {y.mean():.3f}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- Decision Tree (неглубокое — для визуализации) ---
dt = DecisionTreeClassifier(
    max_depth=4,
    min_samples_leaf=50,
    class_weight='balanced',
    random_state=42
)
dt.fit(X_train, y_train)

print("\n=== Decision Tree (глубина 4) ===")
print(export_text(dt, feature_names=FEATURES))

# --- Random Forest ---
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=30,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print("\n=== Random Forest — метрики ===")
print(classification_report(y_test, y_pred,
                             target_names=['Ранение (0)', 'Гибель (1)']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.3f}")

# ─────────────────────────────────────────────
# 10. ВАЖНОСТЬ ПРИЗНАКОВ
# ─────────────────────────────────────────────
print("\n=== Важность признаков ===")

importance_df = pd.DataFrame({
    'feature':    FEATURES,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True)

print(importance_df.head(10).to_string(index=False))

importance_df.to_csv('output_mart/feature_importance.csv',
                     index=False, encoding='utf-8')

# График топ-15
top15 = importance_df.head(15)
fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(top15['feature'][::-1], top15['importance'][::-1], color='steelblue')
ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=9)
ax.set_xlabel('Важность признака (Feature Importance)')
ax.set_title('Топ-15 факторов влияния на тяжесть ДТП\n(Random Forest, 2022–2024)')
ax.set_xlim(0, top15['importance'].max() * 1.15)
plt.tight_layout()
plt.savefig('reports/feature_importance.png', dpi=150)
plt.close()
print("График сохранён: reports/feature_importance.png")

# ─────────────────────────────────────────────
# 11. RISK_SCORE — формула на основе модели
# ─────────────────────────────────────────────
print("\n=== Расчёт risk_score ===")

# Берём топ-7 признаков, нормируем веса
top7 = importance_df.head(7)
weights = dict(zip(
    top7['feature'],
    top7['importance'] / top7['importance'].sum()
))
print("Веса risk_score:")
for feat, w in weights.items():
    print(f"  {feat}: {w:.3f}")

df['risk_score'] = sum(
    pd.to_numeric(df[feat], errors='coerce').fillna(0) * w
    for feat, w in weights.items()
)

# Нормируем в диапазон 0..1
mn, mx = df['risk_score'].min(), df['risk_score'].max()
df['risk_score'] = (df['risk_score'] - mn) / (mx - mn)

print(f"risk_score: min={df['risk_score'].min():.3f}, "
      f"max={df['risk_score'].max():.3f}, "
      f"mean={df['risk_score'].mean():.3f}")

# ─────────────────────────────────────────────
# 12. СОХРАНЕНИЕ ВИТРИН
# ─────────────────────────────────────────────
print("\n=== Сохранение витрин ===")

# mart_dtp_enriched — главная витрина
df.to_csv('output_mart/mart_dtp_enriched.csv', index=False, encoding='utf-8')
print(f"mart_dtp_enriched:      {len(df):,} строк")

# mart_factor_profile — флаги в формате "Да"/"Нет" для DataLens
BOOL_COLS = [c for c in [
    'has_speed_violation', 'has_priority_violation', 'has_alcohol_violation',
    'has_pedestrian_viol', 'has_distance_viol', 'has_maneuver_viol',
    'has_young_driver', 'has_novice_driver',
    'has_truck', 'has_bus', 'has_taxi', 'has_motorcycle', 'has_carsharing',
    'is_night_dtp', 'is_weekend', 'is_rush_hour',
    'has_traffic_light', 'has_crosswalk', 'has_camera', 'is_intersection',
] if c in df.columns]

profile_cols = ['dtp_id', 'year', 'type_group', 'severity',
                'dead_count', 'injured_count', 'risk_score'] + BOOL_COLS
profile = df[[c for c in profile_cols if c in df.columns]].copy()
for col in BOOL_COLS:
    if col in profile.columns:
        profile[col] = pd.to_numeric(profile[col], errors='coerce') \
                         .fillna(0).astype(int).map({1: 'Да', 0: 'Нет'})

profile.to_csv('output_mart/mart_factor_profile.csv', index=False, encoding='utf-8')
print(f"mart_factor_profile:    {len(profile):,} строк")

# mart_dashboard_overview — агрегаты для DataLens
overview_group = [c for c in ['year', 'month', 'district_id', 'type_group']
                  if c in df.columns]
overview = df.groupby(overview_group, dropna=False).agg(
    dtp_count        = ('dtp_id',                 'count'),
    dead_total       = ('dead_count',              'sum'),
    injured_total    = ('injured_count',           'sum'),
    children_dead    = ('dead_children_count',     'sum'),
    children_injured = ('injured_children_count',  'sum'),
    avg_risk_score   = ('risk_score',              'mean'),
).reset_index()

overview.to_csv('output_mart/mart_dashboard_overview.csv', index=False, encoding='utf-8')
print(f"mart_dashboard_overview: {len(overview):,} строк")

# mart_time_dynamics — временная динамика
time_group = [c for c in ['year', 'month', 'season', 'day_of_week_dtp',
                           'time_of_day', 'is_weekend']
              if c in df.columns]

flag_cols_for_pct = {
    'pct_speed_viol':     'has_speed_violation',
    'pct_priority_viol':  'has_priority_violation',
    'pct_alcohol':        'has_alcohol_violation',
    'pct_pedestrian':     'has_pedestrian_viol',
    'pct_night':          'is_night_dtp',
}
# Оставляем только те, что есть в df
flag_cols_for_pct = {k: v for k, v in flag_cols_for_pct.items() if v in df.columns}

time_agg = {
    'dtp_count':    ('dtp_id',       'count'),
    'dead_total':   ('dead_count',   'sum'),
    'injured_total':('injured_count','sum'),
    'pct_fatal':    ('severity',     'mean'),
}
for new_col, src_col in flag_cols_for_pct.items():
    time_agg[new_col] = (src_col, lambda x: pd.to_numeric(x, errors='coerce').mean())

dynamics = df.groupby(time_group, dropna=False).agg(**time_agg).reset_index()
dynamics.to_csv('output_mart/mart_time_dynamics.csv', index=False, encoding='utf-8')
print(f"mart_time_dynamics:      {len(dynamics):,} строк")

# ─────────────────────────────────────────────
# 13. ОТЧЁТ КАЧЕСТВА
# ─────────────────────────────────────────────
print("\n=== Отчёт качества ===")

tables = {
    'mart_dtp_enriched':     df,
    'mart_factor_profile':   profile,
    'mart_dashboard_overview': overview,
    'mart_time_dynamics':    dynamics,
}

with pd.ExcelWriter('reports/final_quality_report.xlsx', engine='openpyxl') as writer:

    # Лист 1: количество строк
    counts = pd.DataFrame([
        {'table': name, 'rows': len(t), 'cols': t.shape[1]}
        for name, t in tables.items()
    ])
    counts.to_excel(writer, sheet_name='row_counts', index=False)

    # Лист 2: пропуски в mart_dtp_enriched
    missing = df.isnull().mean().reset_index()
    missing.columns = ['field', 'pct_missing']
    missing = missing[missing['pct_missing'] > 0] \
                .sort_values('pct_missing', ascending=False)
    missing['pct_missing'] = missing['pct_missing'].round(4)
    missing.to_excel(writer, sheet_name='missing_values', index=False)

    # Лист 3: проверка ключей
    key_check = pd.DataFrame([{
        'check':       'dtp_id уникальны в mart_dtp_enriched',
        'actual':      df['dtp_id'].nunique(),
        'expected':    len(fact_dtp),
        'ok':          df['dtp_id'].nunique() == len(fact_dtp),
    }, {
        'check':       'weather покрытие',
        'actual':      df['temp_c'].notna().sum() if 'temp_c' in df.columns else 0,
        'expected':    len(df),
        'ok':          False,
    }, {
        'check':       'spatial покрытие',
        'actual':      df['cell_id'].notna().sum() if 'cell_id' in df.columns else 0,
        'expected':    len(df),
        'ok':          False,
    }])
    key_check.to_excel(writer, sheet_name='key_checks', index=False)

    # Лист 4: важность признаков (из модели)
    importance_df.to_excel(writer, sheet_name='feature_importance', index=False)

    # Лист 5: severity распределение
    sev = df['severity'].value_counts().reset_index()
    sev.columns = ['severity', 'count']
    sev['label'] = sev['severity'].map({0: 'Ранение', 1: 'Гибель'})
    sev.to_excel(writer, sheet_name='severity_distribution', index=False)

print("reports/final_quality_report.xlsx: ✓")


# ========== 14. СОЗДАЁМ YEAR И НЕДОСТАЮЩИЕ ВИТРИНЫ ==========
print("\n=== Создание недостающих витрин ===")

# Убеждаемся что year есть
if 'year' not in df.columns and 'moment_date' in df.columns:
    df['year'] = df['moment_date'].dt.year
    print("✓ Колонка 'year' создана")

# 1. mart_spatial_risk — агрегаты по H3-ячейкам для карты
if 'cell_id' in df.columns and 'year' in df.columns:
    spatial_risk = df.groupby(['cell_id', 'year'], dropna=False).agg(
        dtp_count=('dtp_id', 'count'),
        fatal_count=('dead_count', 'sum'),
        fatal_rate=('severity', 'mean'),
        avg_risk_score=('risk_score', 'mean'),
        night_share=('is_night_dtp', 'mean') if 'is_night_dtp' in df.columns else None,
        truck_share=('has_truck', 'mean') if 'has_truck' in df.columns else None,
    ).reset_index()
    
    # Убираем None-колонки
    spatial_risk = spatial_risk.dropna(axis=1, how='all')
    spatial_risk.to_csv('output_mart/mart_spatial_risk.csv', index=False, encoding='utf-8')
    print(f"✅ mart_spatial_risk.csv: {len(spatial_risk)} ячеек")
else:
    print("⚠️ mart_spatial_risk: нет cell_id или year")

# 2. mart_weather_context — погодные агрегаты
if 'temp_c' in df.columns and 'year' in df.columns:
    weather_context = df.groupby(['year', 'month', 'season'], dropna=False).agg(
        dtp_count=('dtp_id', 'count'),
        fatal_rate=('severity', 'mean'),
        avg_temp=('temp_c', 'mean'),
        avg_wind=('wind_speed_ms', 'mean') if 'wind_speed_ms' in df.columns else None,
        ice_risk_pct=('ice_risk', 'mean') if 'ice_risk' in df.columns else None,
        bad_weather_pct=('bad_weather_flag', 'mean') if 'bad_weather_flag' in df.columns else None,
    ).reset_index()
    
    weather_context = weather_context.dropna(axis=1, how='all')
    weather_context.to_csv('output_mart/mart_weather_context.csv', index=False, encoding='utf-8')
    print(f"✅ mart_weather_context.csv: {len(weather_context)} строк")
else:
    print("⚠️ mart_weather_context: нет temp_c или year")

# 3. mart_cell_day — ячейка + день недели
if 'cell_id' in df.columns and 'day_of_week_dtp' in df.columns:
    cell_day = df.groupby(['cell_id', 'year', 'month', 'day_of_week_dtp'], dropna=False).agg(
        dtp_count=('dtp_id', 'count'),
        fatal_count=('dead_count', 'sum'),
        avg_risk=('risk_score', 'mean'),
    ).reset_index()
    cell_day.to_csv('output_mart/mart_cell_day.csv', index=False, encoding='utf-8')
    print(f"✅ mart_cell_day.csv: {len(cell_day)} строк")
else:
    print("⚠️ mart_cell_day: нет cell_id или day_of_week_dtp")

# 4. mart_cell_hour — ячейка + час
if 'cell_id' in df.columns and 'hour' in df.columns:
    cell_hour = df.groupby(['cell_id', 'year', 'hour'], dropna=False).agg(
        dtp_count=('dtp_id', 'count'),
        fatal_count=('dead_count', 'sum'),
        avg_risk=('risk_score', 'mean'),
    ).reset_index()
    cell_hour.to_csv('output_mart/mart_cell_hour.csv', index=False, encoding='utf-8')
    print(f"✅ mart_cell_hour.csv: {len(cell_hour)} строк")
else:
    print("⚠️ mart_cell_hour: нет cell_id или hour")



print("\n" + "="*50)
print("ГОТОВО. Файлы сохранены:")
print("  output_mart/mart_dtp_enriched.csv")
print("  output_mart/mart_factor_profile.csv")
print("  output_mart/mart_dashboard_overview.csv")
print("  output_mart/mart_time_dynamics.csv")
print("  output_mart/feature_importance.csv")
print("  reports/feature_importance.png")
print("  reports/final_quality_report.xlsx")
print("="*50)

=== Загрузка данных ===
fact_dtp:  41,131 строк
fact_part: 98,402 строк
fact_veh:  71,589 строк
weather:   4,600 строк  ← покрытие 11.2%
spatial:   24,552 строк  ← покрытие 59.7%

=== Джойн ===
Строк после джойна: 41,131 ✓
Колонок: 118

=== Временные признаки ===
Временные признаки: ✓

Распределение severity:
severity
0    39558
1     1573
Name: count, dtype: int64
Доля с гибелью: 0.038

=== Классификатор типов ДТП ===
type_name
Столкновение                                        18712
Наезд на пешехода                                   13637
Наезд на велосипедиста                               2120
Наезд на препятствие                                 2114
Падение пассажира                                    2021
Наезд на стоящее т/с                                 1573
Опрокидывание                                         789
Съезд с дороги                                         49
Иной вид ДТП                                           35
Наезд на лицо, осуществляющее проведение рабо

In [ ]:
"""
Генерация factor_dictionary.xlsx и README_final.md

"""

import os
import pandas as pd

os.makedirs('output_mart', exist_ok=True)
os.makedirs('reports',     exist_ok=True)

# ─────────────────────────────────────────────
# 1. FACTOR DICTIONARY
# ─────────────────────────────────────────────

factors = [

    # ── ИДЕНТИФИКАТОРЫ ──────────────────────────────────────────────────
    dict(feature='dtp_id',            source='fact_dtp',      dtype='int',
         group='Идентификатор',
         description='Уникальный ID дорожно-транспортного происшествия',
         rule='Исходное поле из выгрузки',
         status='ready'),

    # ── ВРЕМЕННЫЕ ПРИЗНАКИ ──────────────────────────────────────────────
    dict(feature='moment_date',        source='fact_dtp',      dtype='date',
         group='Время',
         description='Дата ДТП',
         rule='Исходное поле, приведено к pd.datetime',
         status='ready'),
    dict(feature='moment_time',        source='fact_dtp',      dtype='time',
         group='Время',
         description='Время ДТП',
         rule='Исходное поле',
         status='ready'),
    dict(feature='hour',               source='fact_dtp',      dtype='int (0–23)',
         group='Время',
         description='Час ДТП',
         rule='pd.to_datetime(moment_time).dt.hour',
         status='ready'),
    dict(feature='month',              source='fact_dtp',      dtype='int (1–12)',
         group='Время',
         description='Месяц ДТП',
         rule='moment_date.dt.month',
         status='ready'),
    dict(feature='day_of_week_dtp',    source='fact_dtp',      dtype='int (0–6)',
         group='Время',
         description='День недели (0=пн, 6=вс)',
         rule='moment_date.dt.dayofweek',
         status='ready'),
    dict(feature='time_of_day',        source='fact_dtp',      dtype='string',
         group='Время',
         description='Укрупнённое время суток: Утро/День/Вечер/Ночь',
         rule='Утро 6–10, День 10–17, Вечер 17–22, Ночь иначе',
         status='ready'),
    dict(feature='is_night_dtp',       source='fact_dtp',      dtype='int (0/1)',
         group='Время',
         description='Флаг ночного времени',
         rule='1 если hour >= 22 или hour < 6, иначе 0',
         status='ready'),
    dict(feature='is_rush_hour',       source='fact_dtp',      dtype='int (0/1)',
         group='Время',
         description='Флаг часа пик',
         rule='1 если hour in 7–10 или 17–20, иначе 0',
         status='ready'),
    dict(feature='season',             source='feat_weather_dtp / fact_dtp', dtype='string',
         group='Время',
         description='Сезон года: Зима/Весна/Лето/Осень',
         rule='Из weather если есть, иначе по месяцу: 12,1,2=Зима и т.д.',
         status='ready'),
    dict(feature='is_weekend',         source='feat_weather_dtp',  dtype='int (0/1)',
         group='Время',
         description='Флаг выходного дня (сб/вс)',
         rule='Из weather; при отсутствии — по day_of_week_dtp in [5,6]',
         status='ready'),

    # ── ТЯЖЕСТЬ И ПОСЛЕДСТВИЯ ───────────────────────────────────────────
    dict(feature='severity',           source='fact_dtp',      dtype='int (0/1)',
         group='Целевая переменная',
         description='Тяжесть ДТП: 1 = была гибель, 0 = только ранения',
         rule='1 если dead_count > 0, иначе 0',
         status='ready'),
    dict(feature='dead_count',         source='fact_dtp',      dtype='int',
         group='Последствия',
         description='Количество погибших',
         rule='Исходное поле',
         status='ready'),
    dict(feature='injured_count',      source='fact_dtp',      dtype='int',
         group='Последствия',
         description='Количество раненых',
         rule='Исходное поле',
         status='ready'),
    dict(feature='dead_children_count',source='fact_dtp',      dtype='int',
         group='Последствия',
         description='Количество погибших детей',
         rule='Исходное поле',
         status='ready'),
    dict(feature='injured_children_count', source='fact_dtp',  dtype='int',
         group='Последствия',
         description='Количество раненых детей',
         rule='Исходное поле',
         status='ready'),

    # ── ТИП ДТП ─────────────────────────────────────────────────────────
    dict(feature='type_code',          source='fact_dtp',      dtype='int',
         group='Тип ДТП',
         description='Исходный код типа ДТП',
         rule='Исходное поле',
         status='ready'),
    dict(feature='type_name',          source='fact_dtp',      dtype='string',
         group='Тип ДТП',
         description='Название типа ДТП из исходных данных',
         rule='Исходное поле',
         status='ready'),
    dict(feature='type_group',         source='fact_dtp',      dtype='string',
         group='Тип ДТП',
         description='Укрупнённая группа типа ДТП (9 групп + Прочее)',
         rule='Маппинг type_name → group_map; не найденные → Прочее',
         status='ready'),

    # ── ФЛАГИ НАРУШЕНИЙ ПДД ─────────────────────────────────────────────
    dict(feature='has_speed_violation',    source='fact_participant', dtype='int (0/1)',
         group='Нарушения ПДД',
         description='Нарушение скоростного режима в ДТП',
         rule='1 если хоть один участник имеет нарушение со словом "скорост"',
         status='ready'),
    dict(feature='has_priority_violation', source='fact_participant', dtype='int (0/1)',
         group='Нарушения ПДД',
         description='Нарушение приоритета / проезд на красный',
         rule='Ключевые слова: приоритет, сигнал, светофор, регулирован',
         status='ready'),
    dict(feature='has_alcohol_violation',  source='fact_participant', dtype='int (0/1)',
         group='Нарушения ПДД',
         description='Алкогольное или наркотическое опьянение',
         rule='Ключевые слова: опьянен, алкогол, наркот, состоянии. '
              'ВНИМАНИЕ: в текущих данных = 0 для всех — возможно иное кодирование поля',
         status='needs_check'),
    dict(feature='has_pedestrian_viol',    source='fact_participant', dtype='int (0/1)',
         group='Нарушения ПДД',
         description='Нарушение правил в отношении пешехода',
         rule='Ключевые слова: пешеход',
         status='ready'),
    dict(feature='has_distance_viol',      source='fact_participant', dtype='int (0/1)',
         group='Нарушения ПДД',
         description='Нарушение дистанции',
         rule='Ключевые слова: дистанц',
         status='ready'),
    dict(feature='has_maneuver_viol',      source='fact_participant', dtype='int (0/1)',
         group='Нарушения ПДД',
         description='Нарушение правил манёвра',
         rule='Ключевые слова: маневр',
         status='ready'),

    # ── ФЛАГИ УЧАСТНИКОВ ────────────────────────────────────────────────
    dict(feature='has_young_driver',   source='fact_participant', dtype='int (0/1)',
         group='Участники',
         description='Водитель до 25 лет',
         rule='1 если хоть один водитель с person_age < 25',
         status='ready'),
    dict(feature='has_novice_driver',  source='fact_participant', dtype='int (0/1)',
         group='Участники',
         description='Водитель-новичок (стаж < 2 лет)',
         rule='1 если хоть один водитель с driver_service_length < 2',
         status='ready'),

    # ── ФЛАГИ ТРАНСПОРТНЫХ СРЕДСТВ ───────────────────────────────────────
    dict(feature='has_truck',          source='fact_vehicle',  dtype='int (0/1)',
         group='Транспорт',
         description='Участие грузового ТС',
         rule='1 если transport_type_name содержит "Грузовой"',
         status='ready'),
    dict(feature='has_bus',            source='fact_vehicle',  dtype='int (0/1)',
         group='Транспорт',
         description='Участие автобуса',
         rule='1 если transport_type_name содержит "Автобус"',
         status='ready'),
    dict(feature='has_taxi',           source='fact_vehicle',  dtype='int (0/1)',
         group='Транспорт',
         description='Участие такси',
         rule='1 если transport_type_name содержит "Такси"',
         status='ready'),
    dict(feature='has_motorcycle',     source='fact_vehicle',  dtype='int (0/1)',
         group='Транспорт',
         description='Участие мотоцикла или мопеда',
         rule='1 если transport_type_name in [Мотоцикл, Мопед]',
         status='ready'),
    dict(feature='has_carsharing',     source='fact_vehicle',  dtype='int (0/1)',
         group='Транспорт',
         description='Участие каршерингового автомобиля',
         rule='1 если transport_type_name содержит "Каршеринг"',
         status='ready'),

    # ── ПОГОДНЫЕ ПРИЗНАКИ ────────────────────────────────────────────────
    dict(feature='temp_c',             source='feat_weather_dtp', dtype='float, °C',
         group='Погода',
         description='Температура воздуха в момент ДТП',
         rule='API Open-Meteo по координатам и времени ДТП. '
              'Покрытие ~11% — остальное заполнено медианой для ML',
         status='needs_check'),
    dict(feature='wind_speed_ms',      source='feat_weather_dtp', dtype='float, м/с',
         group='Погода',
         description='Скорость ветра',
         rule='Open-Meteo, wind_speed_unit=ms',
         status='needs_check'),
    dict(feature='ice_risk',           source='feat_weather_dtp', dtype='int (0/1)',
         group='Погода',
         description='Риск гололёда',
         rule='Правило Студента 2 — уточнить в weather README',
         status='needs_check'),
    dict(feature='bad_weather_flag',   source='feat_weather_dtp', dtype='int (0/1)',
         group='Погода',
         description='Неблагоприятные погодные условия',
         rule='Правило Студента 2 — уточнить в weather README',
         status='needs_check'),
    dict(feature='is_dark',            source='feat_weather_dtp', dtype='int (0/1)',
         group='Погода',
         description='Тёмное время суток по астрономическим данным',
         rule='1 если момент ДТП до рассвета или после заката',
         status='ready'),

    # ── ПРОСТРАНСТВЕННЫЕ ПРИЗНАКИ ────────────────────────────────────────
    dict(feature='cell_id',            source='feat_spatial_dtp', dtype='string',
         group='Гео',
         description='H3-ячейка (разрешение 8) для агрегации',
         rule='h3_r8 из feat_spatial_dtp. Покрытие ~60%',
         status='ready'),
    dict(feature='is_intersection',    source='feat_spatial_dtp', dtype='int (0/1)',
         group='Гео',
         description='ДТП на перекрёстке',
         rule='Исходное поле из spatial',
         status='ready'),
    dict(feature='is_open_road',       source='feat_spatial_dtp', dtype='int (0/1)',
         group='Гео',
         description='ДТП на открытом участке дороги (не перекрёсток)',
         rule='Исходное поле из spatial',
         status='ready'),
    dict(feature='has_traffic_light',  source='feat_spatial_dtp', dtype='int (0/1)',
         group='Гео',
         description='Наличие светофора вблизи места ДТП',
         rule='Исходное поле из spatial. Покрытие ~56%',
         status='ready'),
    dict(feature='has_crosswalk',      source='feat_spatial_dtp', dtype='int (0/1)',
         group='Гео',
         description='Наличие пешеходного перехода вблизи места ДТП',
         rule='Исходное поле из spatial. Покрытие ~56%',
         status='ready'),
    dict(feature='has_camera',         source='feat_spatial_dtp', dtype='int (0/1)',
         group='Гео',
         description='Наличие камеры фотовидеофиксации',
         rule='Исходное поле из spatial. Покрытие ~60%',
         status='ready'),
    dict(feature='has_bus_stop',       source='feat_spatial_dtp', dtype='int (0/1)',
         group='Гео',
         description='Наличие остановки ОТ вблизи места ДТП',
         rule='Исходное поле из spatial',
         status='ready'),
    dict(feature='distance_to_bus_stop_m', source='feat_spatial_dtp', dtype='float, м',
         group='Гео',
         description='Расстояние до ближайшей остановки ОТ',
         rule='Исходное поле из spatial. Покрытие ~56%',
         status='ready'),
    dict(feature='distance_to_intersection_m', source='feat_spatial_dtp', dtype='float, м',
         group='Гео',
         description='Расстояние до ближайшего перекрёстка',
         rule='Исходное поле из spatial. Покрытие ~56%',
         status='ready'),
    dict(feature='poi_count_100m',     source='feat_spatial_dtp', dtype='int',
         group='Гео',
         description='Количество объектов притяжения (POI) в радиусе 100м',
         rule='Исходное поле из spatial',
         status='ready'),
    dict(feature='traffic_lane_amount',source='feat_spatial_dtp', dtype='int',
         group='Гео',
         description='Количество полос движения',
         rule='Исходное поле из spatial',
         status='ready'),
    dict(feature='lane_width_m',       source='feat_spatial_dtp', dtype='float, м',
         group='Гео',
         description='Ширина полосы движения',
         rule='Исходное поле из spatial',
         status='ready'),
    dict(feature='is_multilane',       source='feat_spatial_dtp', dtype='int (0/1)',
         group='Гео',
         description='Многополосная дорога',
         rule='Исходное поле из spatial',
         status='ready'),

    # ── ML И ПРОИЗВОДНЫЕ ────────────────────────────────────────────────
    dict(feature='risk_score',         source='build_marts.py', dtype='float (0–1)',
         group='ML / Производные',
         description='Индекс риска летального исхода ДТП',
         rule='Взвешенная сумма топ-7 признаков по feature_importances_ '
              'Random Forest, нормированная в [0, 1]. '
              'Веса: type_code=0.258, hour=0.191, has_truck=0.150, '
              'is_night_dtp=0.132, distance_to_bus_stop_m=0.117, '
              'has_distance_viol=0.083, has_speed_violation=0.068',
         status='ready'),
]

fd = pd.DataFrame(factors, columns=[
    'feature', 'group', 'source', 'dtype',
    'description', 'rule', 'status'
])

# Цветовая легенда статусов для Excel
STATUS_COLORS = {
    'ready':          'C6EFCE',  # зелёный
    'needs_check':    'FFEB9C',  # жёлтый
    'missing_source': 'FFC7CE',  # красный
    'deprecated':     'D9D9D9',  # серый
}

from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

out_path = 'output_mart/factor_dictionary.xlsx'
fd.to_excel(out_path, index=False, sheet_name='factor_dictionary')

wb = load_workbook(out_path)
ws = wb.active

# Ширина колонок
col_widths = {
    'A': 35,   # feature
    'B': 20,   # group
    'C': 22,   # source
    'D': 18,   # dtype
    'E': 45,   # description
    'F': 60,   # rule
    'G': 15,   # status
}
for col_letter, width in col_widths.items():
    ws.column_dimensions[col_letter].width = width

# Заголовок — жирный
header_font = Font(bold=True, size=11)
for cell in ws[1]:
    cell.font = header_font
    cell.alignment = Alignment(horizontal='center', wrap_text=True)

# Цвет по статусу (колонка G = 7)
thin = Side(style='thin', color='AAAAAA')
border = Border(left=thin, right=thin, top=thin, bottom=thin)

for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
    status_cell = row[6]  # колонка G
    status_val  = str(status_cell.value or '').strip()
    fill_color  = STATUS_COLORS.get(status_val, 'FFFFFF')
    fill        = PatternFill('solid', fgColor=fill_color)
    for cell in row:
        cell.fill      = fill
        cell.border    = border
        cell.alignment = Alignment(wrap_text=True, vertical='top')

# Второй лист — легенда статусов
ws2 = wb.create_sheet('legend')
ws2.append(['Статус', 'Смысл'])
ws2.append(['ready',          'Поле готово, проверено, используется'])
ws2.append(['needs_check',    'Поле есть, но требует проверки или имеет ограниченное покрытие'])
ws2.append(['missing_source', 'Источник недоступен, поле пустое (NaN)'])
ws2.append(['deprecated',     'Поле устарело, не используется'])

for i, (status, color) in enumerate(STATUS_COLORS.items(), start=2):
    ws2.cell(row=i, column=1).fill = PatternFill('solid', fgColor=color)
ws2.column_dimensions['A'].width = 20
ws2.column_dimensions['B'].width = 50

wb.save(out_path)
print(f"factor_dictionary.xlsx: {len(fd)} признаков ✓")


factor_dictionary.xlsx: 51 признаков ✓
README_final.md: ✓

=== ДОКУМЕНТАЦИЯ ГОТОВА ===
  output_mart/factor_dictionary.xlsx
  README_final.md
